# Sudoku Solver Demo - Adaptive Inference in Masked Diffusion Models

**Nhom 5 - Intro to AI Pilot Study**

Notebook tuong tac truc quan hoa qua trinh giai Sudoku theo tung buoc,
so sanh cac chien luoc: Random, Top Probability, Margin + Gumbel noise,
CSP co dien (Backtracking, MRV+FC, AC-3).

> **Ngay 2:** Chay voi `DummyMDM` + mock data.
> **Ngay 3+:** Chuyen co `USE_REAL_MODEL = True` de dung `TinyMDM` + `adaptive_decode`.

## Setup cho Google Colab
Hai cell duoi day chi can chay khi mo notebook tren **Google Colab**.
Neu chay local, chung se tu bo qua.

In [ ]:
import os, sys

# --- Google Colab: Mount Google Drive ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Khong chay tren Colab - bo qua mount Drive.")

SHARED_DRIVE_DIR = '/content/drive/MyDrive/IntroAI_Nhom5_Shared'

if IN_COLAB and os.path.isdir(SHARED_DRIVE_DIR):
    print(f"Shared Drive: {SHARED_DRIVE_DIR}")
elif IN_COLAB:
    print(f"Chua tim thay: {SHARED_DRIVE_DIR}")
    print("   Hay tao thu muc nay tren Drive va share voi nhom.")

In [ ]:
%load_ext autoreload
%autoreload 2

REPO_URL = 'https://github.com/<YOUR_ORG>/intro-ai-project-group3.git'  # TODO: cap nhat URL
REPO_DIR = '/content/intro-ai-project-group3'

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    else:
        %cd {REPO_DIR}
        !git pull
    sys.path.insert(0, REPO_DIR)
    %cd {REPO_DIR}
    !pip install -q -r requirements.txt
else:
    _cwd = os.getcwd()
    _root = (os.path.abspath(os.path.join(_cwd, '..'))
             if os.path.basename(_cwd) == 'notebooks' else _cwd)
    if _root not in sys.path:
        sys.path.insert(0, _root)
    print(f"Project root: {_root}")

## Import modules

In [ ]:
import numpy as np
import torch
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

from src.interfaces import (
    DummyMDM,
    generate_mock_data,
    generate_mock_step_history,
    validate_step_history,
)
from src.visualizer import render_board_html, render_step_history

print("Import thanh cong!")

## Cau hinh che do chay

| Co | Gia tri | Y nghia |
|---|---|---|
| `USE_REAL_MODEL` | `False` | Ngay 2: dung `DummyMDM` + mock data |
| | `True` | Ngay 3+: dung `TinyMDM` + `adaptive_decode` |

In [ ]:
# CAU HINH - Doi USE_REAL_MODEL = True o Ngay 3

USE_REAL_MODEL = False    # <-- Ngay 3: doi thanh True

N_SIZE      = 9           # Kich thuoc bang: 4, 6, hoac 9
NUM_SAMPLES = 5           # So bang mau
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Mode   : {'Real Model' if USE_REAL_MODEL else 'Mock (DummyMDM)'}")
print(f"Board  : {N_SIZE}x{N_SIZE} | Samples: {NUM_SAMPLES} | Device: {DEVICE}")

## Nap du lieu & Giai bang

In [ ]:
# --- 1. Nap du lieu & Model ---
if USE_REAL_MODEL:
    from src.data_csp import load_pilot_data
    from src.models import TinyMDM
    from src.inference import adaptive_decode

    DATA_PATH  = os.path.join(SHARED_DRIVE_DIR, 'sudoku_pilot.npz')
    MODEL_PATH = os.path.join(SHARED_DRIVE_DIR, 'tiny_mdm.pt')

    if os.path.exists(DATA_PATH):
        quizzes, solutions = load_pilot_data(DATA_PATH)
        quizzes, solutions = quizzes[:NUM_SAMPLES], solutions[:NUM_SAMPLES]
        print(f"✅ Đã nạp {len(quizzes)} bảng từ {DATA_PATH}")
    else:
        print(f"⚠️ Không tìm thấy {DATA_PATH}, fallback sang mock data...")
        quizzes, solutions = generate_mock_data(num_samples=NUM_SAMPLES, n_size=N_SIZE)

    model = TinyMDM(vocab_size=N_SIZE + 1, seq_len=N_SIZE ** 2)
    if os.path.exists(MODEL_PATH):
        model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
        print(f"✅ Đã nạp checkpoint: {MODEL_PATH}")
    else:
        print(f"⚠️ Không tìm thấy checkpoint {MODEL_PATH}. Chạy thử nghiệm với TinyMDM khởi tạo ngẫu nhiên.")
    model.to(DEVICE).eval()
else:
    quizzes, solutions = generate_mock_data(num_samples=NUM_SAMPLES, n_size=N_SIZE)
    model = DummyMDM(vocab_size=N_SIZE + 1)
    print(f"Sinh {NUM_SAMPLES} bảng giả lập {N_SIZE}x{N_SIZE}")

# --- 2. Giải tất cả các bảng & Ghi nhận lịch sử ---
all_histories = []
all_preds     = []

for i in range(len(quizzes)):
    quiz = quizzes[i]
    if USE_REAL_MODEL:
        pred, history = adaptive_decode(
            model, quiz, strategy="margin",
            steps=N_SIZE ** 2, device=DEVICE,
        )
    else:
        history = generate_mock_step_history(quiz, n_size=N_SIZE)
        pred = history[-1]["board_state"].copy() if history else quiz.copy()

    # Kiểm tra Data Contract Gate
    assert validate_step_history(history, seq_len=N_SIZE ** 2), \
        f"Bảng #{i+1}: step_history không hợp lệ!"

    all_histories.append(history)
    all_preds.append(pred)

mode_label = "Real Model" if USE_REAL_MODEL else "Mock (DummyMDM)"
print(f"\n✅ Đã giải xong {len(quizzes)} bảng [{mode_label}]!")
for i in range(len(quizzes)):
    n_mask = int(np.sum(quizzes[i] == 0))
    print(f"   Bảng #{i+1}: {n_mask} ô trống -> {len(all_histories[i])} bước giải")

## Demo tuong tac: Step-by-Step Visualization

Dung **thanh truot** de di chuyen qua tung buoc, hoac nhan **Play** de xem tu dong.

| Mau | Y nghia |
|---|---|
| Xanh dam | O de bai cho truoc |
| Xanh la | O vua mo - tu tin cao (margin >= 0.5) |
| Cam | O vua mo - tu tin thap (margin < 0.5) |
| Xam nhat | O da mo o buoc truoc |
| Trang | O con trong [MASK] |

In [ ]:
# INTERACTIVE DEMO - Sudoku Step-by-Step

# --- Output area ---
out = widgets.Output()

# --- Puzzle selector ---
puzzle_opts = [
    (f"Bang #{i+1} ({int(np.sum(quizzes[i]==0))} trong, "
     f"{len(all_histories[i])} buoc)", i)
    for i in range(len(quizzes))
]
w_puzzle = widgets.Dropdown(
    options=puzzle_opts, value=0,
    description='Bang:',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='360px'),
)

# --- Step slider ---
_max0 = max(len(all_histories[0]) - 1, 0)
w_step = widgets.IntSlider(
    value=0, min=0, max=_max0,
    description='Buoc:',
    continuous_update=False,
    style={'description_width': '50px'},
    layout=widgets.Layout(width='420px'),
)

# --- Play / Pause ---
w_play = widgets.Play(
    value=0, min=0, max=_max0,
    step=1, interval=300,
    description='Play', disabled=False,
)
widgets.jslink((w_play, 'value'), (w_step, 'value'))

# --- Speed ---
w_speed = widgets.IntSlider(
    value=300, min=50, max=1500, step=50,
    description='ms/buoc:',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='220px'),
)

def _on_speed(change):
    w_play.interval = change['new']
w_speed.observe(_on_speed, names='value')

# --- Legend ---
w_legend = widgets.HTML(
    '<div style="font-family:sans-serif;font-size:12px;margin:4px 0;">'
    '<span style="background:#1a5276;color:#fff;padding:2px 8px;border-radius:3px;'
    'margin-right:6px;">De bai</span>'
    '<span style="background:#27ae60;color:#fff;padding:2px 8px;border-radius:3px;'
    'margin-right:6px;">Mo moi (cao)</span>'
    '<span style="background:#e67e22;color:#fff;padding:2px 8px;border-radius:3px;'
    'margin-right:6px;">Mo moi (thap)</span>'
    '<span style="background:#d5dbdb;color:#2c3e50;padding:2px 8px;border-radius:3px;'
    'margin-right:6px;">Da mo</span>'
    '<span style="background:#fdfefe;color:#bdc3c7;padding:2px 8px;border-radius:3px;'
    'border:1px solid #eee;">Trong</span></div>'
)


def _update(change=None):
    pidx = w_puzzle.value
    hist = all_histories[pidx]
    quiz = quizzes[pidx]
    if not hist:
        with out:
            clear_output(wait=True)
            display(HTML("<p>Bang khong co o trong.</p>"))
        return

    sidx = min(w_step.value, len(hist) - 1)
    item = hist[sidx]

    with out:
        clear_output(wait=True)
        html = render_board_html(quiz, item, n_size=N_SIZE)

        total = len(hist)
        pct   = (sidx + 1) / total * 100
        filled = int(np.sum(item['board_state'] != 0))
        L = N_SIZE * N_SIZE

        bar = (
            '<div style="font-family:sans-serif;margin:8px 10px;max-width:500px;">'
            '<div style="background:#ecf0f1;border-radius:4px;height:6px;overflow:hidden;">'
            f'<div style="background:linear-gradient(90deg,#3498db,#2ecc71);'
            f'height:6px;border-radius:4px;width:{pct:.1f}%;"></div></div>'
            f'<div style="color:#7f8c8d;font-size:11px;margin-top:4px;text-align:center;">'
            f'Buoc {sidx+1}/{total} - {filled}/{L} o da dien</div></div>'
        )
        display(HTML(html + bar))


def _on_puzzle(change):
    pidx = change['new']
    new_max = max(len(all_histories[pidx]) - 1, 0)
    w_step.max = new_max
    w_step.value = 0
    w_play.max = new_max
    w_play.value = 0
    _update()

w_puzzle.observe(_on_puzzle, names='value')
w_step.observe(_update, names='value')

# --- Assemble UI ---
row1 = widgets.HBox([w_puzzle, w_speed])
row2 = widgets.HBox([w_play, w_step])
ui = widgets.VBox([
    widgets.HTML(
        "<h3 style='font-family:sans-serif;margin-bottom:4px;'>"
        "Sudoku Solver - Step-by-Step Visualization</h3>"
    ),
    w_legend, row1, row2, out,
], layout=widgets.Layout(padding='10px'))

display(ui)
_update()

## Chay animation tuan tu (thay the)
Cell duoi day phat toan bo qua trinh giai bang dau tien dang animation.
Phu hop khi muon xem nhanh khong can widget.

In [ ]:
# Uncomment de chay animation tuan tu:
# render_step_history(quizzes[0], all_histories[0], n_size=N_SIZE, delay=0.15)